# Steam — Exploratory Data Analysis

Canonical dataset at `data/canonical/steam/v1/`.  
**Key difference from ML-1M**: this is a library *snapshot* (no timestamps), signal is ownership + playtime, not explicit ratings.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DATA_DIR = Path("../data/canonical/steam/v1")

users        = pd.read_parquet(DATA_DIR / "users.parquet")
items        = pd.read_parquet(DATA_DIR / "items.parquet")
interactions = pd.read_parquet(DATA_DIR / "interactions.parquet")

with open(DATA_DIR / "manifest.json") as f:
    manifest = json.load(f)

## 1  Dataset overview

In [ ]:
n_users = users["user_id"].nunique()
n_items = items["item_id"].nunique()
n_interactions = len(interactions)
density = n_interactions / (n_users * n_items)

print(f"Users        : {n_users:,}")
print(f"Items        : {n_items:,}")
print(f"Interactions : {n_interactions:,}")
print(f"Density      : {density:.4%}")
print()
print("Standard targets:")
for k, v in manifest["standard_targets"].items():
    print(f"  {k}: {v}")
print()
print("Caveats:")
for c in manifest["caveats"]:
    print(f"  - {c}")

In [ ]:
print("Interactions schema:")
interactions.dtypes.to_frame("dtype")

## 2  Playtime distributions

Playtime is the primary engagement signal. It is heavily zero-inflated and right-skewed.

In [ ]:
pt = interactions["playtime_forever"]
zero_frac = (pt == 0).mean()
print(f"Zero-playtime rows : {(pt == 0).sum():,}  ({zero_frac:.1%} of all owned entries)")
print()
print(pt.describe().map(lambda x: f"{x:,.1f}"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# All playtime (log1p scale)
axes[0].hist(np.log1p(pt), bins=80, color="steelblue", edgecolor="none")
axes[0].set_title("playtime_forever — log1p scale")
axes[0].set_xlabel("log1p(minutes)")
axes[0].set_ylabel("# interactions")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))

# Played-only playtime (log scale)
played = pt[pt > 0]
axes[1].hist(np.log10(played), bins=80, color="steelblue", edgecolor="none")
axes[1].set_title(f"playtime > 0 only  (n={len(played):,})")
axes[1].set_xlabel("log10(minutes)")
axes[1].set_ylabel("# interactions")
ticks = [1, 10, 60, 600, 6000, 60000]
axes[1].set_xticks(np.log10(ticks))
axes[1].set_xticklabels([f"{t}m" if t < 60 else f"{t//60}h" for t in ticks], fontsize=8)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))

# playtime_2weeks (log1p)
pt2 = interactions["playtime_2weeks"]
axes[2].hist(np.log1p(pt2[pt2 > 0]), bins=60, color="steelblue", edgecolor="none")
axes[2].set_title(f"playtime_2weeks > 0  (n={( pt2 > 0).sum():,})")
axes[2].set_xlabel("log1p(minutes)")
axes[2].set_ylabel("# interactions")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))

plt.tight_layout()
plt.show()

## 3  Target distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# target_played_120
p120 = interactions["target_played_120"].value_counts().sort_index()
axes[0].bar(["Not played 2h+", "Played 2h+"], p120.values,
            color=["#d9534f", "#5cb85c"], edgecolor="white")
axes[0].set_title("target_played_120 (played ≥ 120 min)")
axes[0].set_ylabel("# interactions")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
pct120 = p120[1] / p120.sum()
axes[0].text(1, p120[1] + 20000, f"{pct120:.1%}", ha="center", fontsize=11)

# target_playtime quantiles
quantiles = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
qt_vals = interactions["target_playtime"].quantile(quantiles)
axes[1].bar([f"p{int(q*100)}" for q in quantiles], qt_vals.values,
            color="steelblue", edgecolor="white")
axes[1].set_title("target_playtime quantiles (minutes)")
axes[1].set_ylabel("minutes")
for bar, val in zip(axes[1].patches, qt_vals.values):
    label = f"{val:.0f}m" if val < 120 else f"{val/60:.0f}h"
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 label, ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 4  Item metadata coverage

In [ ]:
meta_coverage = items["metadata_available"].value_counts()
print(f"Items with metadata    : {meta_coverage.get(True, 0):,}")
print(f"Items without metadata : {meta_coverage.get(False, 0):,}")
print()

# How many interactions involve items without metadata?
no_meta_items = set(items.loc[~items["metadata_available"], "item_id"])
no_meta_interactions = interactions["item_id"].isin(no_meta_items).sum()
print(f"Interactions for items without metadata: {no_meta_interactions:,} ({no_meta_interactions/len(interactions):.1%})")

## 5  Genre and tag analysis

Genres and tags are stored as JSON arrays in the items table. Tags are richer and noisier than genres.

In [ ]:
meta_items = items[items["metadata_available"]].copy()

def parse_json_column(series, n=30):
    return (
        series
        .dropna()
        .apply(lambda x: json.loads(x) if isinstance(x, str) and x.strip() else [])
        .explode()
        .str.strip()
        .value_counts()
        .head(n)
    )

genre_counts = parse_json_column(meta_items["genres_json"])
tag_counts   = parse_json_column(meta_items["tags_json"], n=40)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(genre_counts.index[::-1], genre_counts.values[::-1],
             color="steelblue", edgecolor="white")
axes[0].set_title("Top genres (items with metadata)")
axes[0].set_xlabel("# items")

axes[1].barh(tag_counts.index[::-1], tag_counts.values[::-1],
             color="steelblue", edgecolor="white")
axes[1].set_title("Top 40 tags (items with metadata)")
axes[1].set_xlabel("# items")

plt.tight_layout()
plt.show()

In [ ]:
# Price distribution (items with metadata and non-empty price)
prices = (
    meta_items["price_raw"]
    .dropna()
    .str.replace("$", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
    .dropna()
)
print(f"Items with parseable price: {len(prices):,}")
print(f"Free (price=0): {(prices == 0).sum():,}")
print(prices[prices > 0].describe().map(lambda x: f"{x:.2f}"))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(prices[prices > 0], bins=60, color="steelblue", edgecolor="none")
ax.set_title("Price distribution (paid items only)")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("# items")
plt.tight_layout()
plt.show()

In [ ]:
# Release year distribution
years = (
    meta_items["release_date"]
    .dropna()
    .str[:4]
    .pipe(pd.to_numeric, errors="coerce")
    .dropna()
    .astype(int)
)
year_counts = years.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(year_counts.index, year_counts.values, color="steelblue", edgecolor="none", width=0.8)
ax.set_title("Items by release year")
ax.set_xlabel("Year")
ax.set_ylabel("# items")
plt.tight_layout()
plt.show()

## 6  User behavior — library size

In [ ]:
per_user = interactions.groupby("user_id").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(np.log10(per_user), bins=80, color="steelblue", edgecolor="none")
axes[0].set_title("Items owned per user (log10 scale)")
axes[0].set_xlabel("log10(# items owned)")
axes[0].set_ylabel("# users")
ticks = [1, 5, 10, 50, 100, 500, 1000]
axes[0].set_xticks(np.log10(ticks))
axes[0].set_xticklabels(ticks, fontsize=8)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))

# reported_items_count vs actual interaction count
user_actual = per_user.rename("actual").reset_index()
user_merged = users.merge(user_actual, on="user_id")
axes[1].scatter(
    user_merged["reported_items_count"],
    user_merged["actual"],
    alpha=0.05, s=2, color="steelblue",
)
axes[1].set_title("Reported vs actual library size")
axes[1].set_xlabel("reported_items_count")
axes[1].set_ylabel("actual owned items")
lim = max(user_merged["reported_items_count"].max(), user_merged["actual"].max())
axes[1].plot([0, lim], [0, lim], "r--", linewidth=0.8, label="y=x")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Items owned per user:")
print(per_user.describe().map(lambda x: f"{x:,.1f}"))

## 7  Item popularity — long-tail

In [ ]:
per_item = interactions.groupby("item_id").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(np.log10(per_item), bins=80, color="steelblue", edgecolor="none", log=True)
axes[0].set_title("Owners per item (log-log)")
axes[0].set_xlabel("log10(# owners)")
axes[0].set_ylabel("# items (log scale)")
ticks = [1, 10, 100, 1000, 10000]
axes[0].set_xticks(np.log10(ticks))
axes[0].set_xticklabels(ticks, fontsize=8)

# Items with zero owners (in items table but not in interactions)
items_with_interactions = set(interactions["item_id"].unique())
cold_items = items[~items["item_id"].isin(items_with_interactions)]

# Lorenz-like: cumulative share of interactions by item popularity
sorted_counts = per_item.sort_values(ascending=False)
cum_share = sorted_counts.cumsum() / sorted_counts.sum()
item_pct = np.linspace(0, 100, len(cum_share))
axes[1].plot(item_pct, cum_share.values * 100, color="steelblue")
axes[1].set_title("Cumulative interaction share by item rank")
axes[1].set_xlabel("% of items (most popular first)")
axes[1].set_ylabel("% of interactions")
axes[1].axvline(1, color="#d9534f", linestyle="--", linewidth=0.9)
at1pct = cum_share.iloc[int(len(cum_share) * 0.01)]
axes[1].text(1.5, at1pct * 100 - 5, f"top 1% items → {at1pct:.0%} of interactions",
             fontsize=9, color="#d9534f")

plt.tight_layout()
plt.show()

print(f"Items with no owners : {len(cold_items):,}")
print("Owners per item:")
print(per_item.describe().map(lambda x: f"{x:,.1f}"))

## 8  Top games — by ownership and by total playtime

In [ ]:
item_stats = (
    interactions
    .groupby("item_id")
    .agg(
        n_owners=("user_id", "count"),
        total_playtime=("playtime_forever", "sum"),
        median_playtime=("playtime_forever", "median"),
        pct_played_120=("target_played_120", "mean"),
    )
    .reset_index()
    .merge(items[["item_id", "title"]], on="item_id", how="left")
)

print("Top 15 by ownership:")
display_cols = ["title", "n_owners", "pct_played_120", "median_playtime"]
top_own = item_stats.nlargest(15, "n_owners")[display_cols].reset_index(drop=True)
top_own["pct_played_120"] = top_own["pct_played_120"].map("{:.1%}".format)
top_own["median_playtime"] = top_own["median_playtime"].map(lambda x: f"{x:.0f}m")
print(top_own.to_string(index=False))

print()
print("Top 15 by total playtime:")
top_pt = item_stats.nlargest(15, "total_playtime")[["title", "n_owners", "total_playtime"]].reset_index(drop=True)
top_pt["total_playtime"] = top_pt["total_playtime"].map(lambda x: f"{x/60/1000:.0f}k h")
print(top_pt.to_string(index=False))

## 9  Zero-playtime analysis

Owned-but-never-played entries are structurally different from played entries and affect the target definitions.

In [ ]:
# Per-user: what fraction of owned games has zero playtime?
user_zero_frac = (
    interactions
    .assign(is_zero=(interactions["playtime_forever"] == 0))
    .groupby("user_id")["is_zero"]
    .mean()
)

# Per-item: fraction of owners who never played
item_zero_frac = (
    interactions
    .assign(is_zero=(interactions["playtime_forever"] == 0))
    .groupby("item_id")["is_zero"]
    .mean()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(user_zero_frac, bins=50, color="steelblue", edgecolor="none")
axes[0].set_title("Per-user fraction of unplayed owned games")
axes[0].set_xlabel("Fraction with playtime = 0")
axes[0].set_ylabel("# users")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
axes[0].axvline(user_zero_frac.median(), color="#d9534f", linestyle="--", linewidth=1,
                label=f"median={user_zero_frac.median():.0%}")
axes[0].legend(fontsize=9)

axes[1].hist(item_zero_frac, bins=50, color="steelblue", edgecolor="none")
axes[1].set_title("Per-item fraction of owners who never played")
axes[1].set_xlabel("Fraction with playtime = 0")
axes[1].set_ylabel("# items")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
axes[1].axvline(item_zero_frac.median(), color="#d9534f", linestyle="--", linewidth=1,
                label=f"median={item_zero_frac.median():.0%}")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()